In [1]:
import pandas as pd
import numpy as np

In [2]:
poverty = pd.read_csv('/Users/ankitakhatri/Downloads/Poverty2023.csv', encoding='latin-1')
education = pd.read_csv('/Users/ankitakhatri/Downloads/Education2023.csv', encoding='latin-1')
unemployment = pd.read_csv('/Users/ankitakhatri/Downloads/Unemployment2023.csv', encoding='latin-1')
population = pd.read_csv('/Users/ankitakhatri/Downloads/PopulationEstimates.csv', encoding='latin-1')

In [3]:
poverty['Dataset'] = 'poverty'
poverty.head()


,FIPS_Code,Stabr,Area_Name,Attribute,Value,Dataset
0,0,US,United States,POVALL_2023,40763043.0,poverty
1,0,US,United States,CI90LBALL_2023,40485829.0,poverty
2,0,US,United States,CI90UBALL_2023,41040257.0,poverty
3,0,US,United States,PCTPOVALL_2023,12.5,poverty
4,0,US,United States,CI90LBALLP_2023,12.4,poverty


In [4]:
education['Dataset'] = 'education'
education


,FIPS Code,State,Area name,Attribute,Value,Dataset
0,0,US,United States,"Less than a high school diploma, 1970",5.237331e+07,education
1,0,US,United States,"High school diploma only, 1970",3.415805e+07,education
2,0,US,United States,"Some college (1-3 years), 1970",1.165073e+07,education
3,0,US,United States,"Four years of college or higher, 1970",1.171727e+07,education
4,0,US,United States,Percent of adults with less than a high school...,4.770000e+01,education
...,...,...,...,...,...,...
169240,72153,PR,Yauco Municipio,"Bachelor's degree or higher, 2019-23",6.874000e+03,education
169241,72153,PR,Yauco Municipio,Percent of adults who are not high school grad...,2.048521e+01,education
169242,72153,PR,Yauco Municipio,Percent of adults who are high school graduate...,3.396055e+01,education
169243,72153,PR,Yauco Municipio,Percent of adults completing some college or a...,1.843787e+01,education


In [5]:
unemployment['Dataset'] = 'unemployment'
unemployment.head()


,FIPS_Code,State,Area_Name,Attribute,Value,Dataset
0,0,US,United States,Civilian_labor_force_2000,142601576.0,unemployment
1,0,US,United States,Employed_2000,136904853.0,unemployment
2,0,US,United States,Unemployed_2000,5696723.0,unemployment
3,0,US,United States,Unemployment_rate_2000,4.0,unemployment
4,0,US,United States,Civilian_labor_force_2001,143786537.0,unemployment


In [6]:
population['Dataset'] = 'population'
population.head()


,FIPStxt,State,Area_Name,Attribute,Value,Dataset
0,0,US,United States,CENSUS_2020_POP,331449281.0,population
1,0,US,United States,ESTIMATES_BASE_2020,331464948.0,population
2,0,US,United States,POP_ESTIMATE_2020,331526933.0,population
3,0,US,United States,POP_ESTIMATE_2021,332048977.0,population
4,0,US,United States,POP_ESTIMATE_2022,333271411.0,population


In [7]:
# rename columns to match a standard format
poverty = poverty.rename(columns={'Stabr': 'State'})
education = education.rename(columns={'FIPS Code': 'FIPS_Code', 'Area name': 'Area_Name'})
population = population.rename(columns={'FIPStxt': 'FIPS_Code'})

In [8]:
# combine all dfs
combined_data = pd.concat([poverty, education, unemployment, population], ignore_index=True)



In [9]:
# data cleaning

# remove all other states except california
combined_data = combined_data[combined_data['State'] == "CA"]

# remove general california data, keep county data
combined_data = combined_data[combined_data['Area_Name'] != "California"]

# remove data not from 2023 - changed logic, see below
# regex_other_years = r'(?<!\d)(?!2023(?!\d))\d+'
# combined_data = combined_data[~combined_data['Attribute'].str.contains(regex_other_years, regex=True, na=False)]

In [10]:
# filtering logic:
# keep rows that contain '23' or '2023'
# remove rows that contain numbers but do not contain '23'
# keep rows with no numbers

# check if there is a number present
has_number = combined_data['Attribute'].str.contains(r'\d', na=False)

# Check if '23' or '2023' is present
is_2023_context = combined_data['Attribute'].str.contains(r'2023|\b23\b', na=False)

# Keep the row if: (No numbers exist) OR (It is a 2023-related number)
combined_data = combined_data[~has_number | is_2023_context]

# Reset index for cleanliness
combined_data = combined_data.reset_index(drop=True)

# Verify results
print(combined_data.head())

   FIPS_Code State       Area_Name                        Attribute     Value  \
0       6001    CA  Alameda County  Rural_Urban_Continuum_Code_2023       1.0   
1       6001    CA  Alameda County                      POVALL_2023  151872.0   
2       6001    CA  Alameda County                   CI90LBALL_2023  138959.0   
3       6001    CA  Alameda County                   CI90UBALL_2023  164785.0   
4       6001    CA  Alameda County                   PCTPOVALL_2023       9.5   

   Dataset  
0  poverty  
1  poverty  
2  poverty  
3  poverty  
4  poverty  


In [11]:
# education only has rural urban continuum code, why?
education = education[education['State'] == "CA"]
education = education[education['Area_Name'] != "California"]
education.to_csv('/Users/ankitakhatri/Downloads/education.csv', index=False)

# columns that said 2019-23 were removed
# so we fixed the code above accordinly


In [12]:

combined_data.to_csv('/Users/ankitakhatri/Downloads/CombinedData.csv', index=False)

combined_data.head()

,FIPS_Code,State,Area_Name,Attribute,Value,Dataset
0,6001,CA,Alameda County,Rural_Urban_Continuum_Code_2023,1.0,poverty
1,6001,CA,Alameda County,POVALL_2023,151872.0,poverty
2,6001,CA,Alameda County,CI90LBALL_2023,138959.0,poverty
3,6001,CA,Alameda County,CI90UBALL_2023,164785.0,poverty
4,6001,CA,Alameda County,PCTPOVALL_2023,9.5,poverty
